# MATHIVA — Fine-tune flan-t5-base (Phase 2)

Runs on a **free Colab T4 GPU**. Self-contained: upload three small files,
train, then download the model.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

Files to have ready on your machine (from `A:\mathiva\ml\t5\`):
`data/train.jsonl`, `data/val.jsonl`, and `train.py`.

The dataset is the 4-subject standalone-tutor set (140 train / 22 val), so each
example is `question → answer` — the model learns to answer Senior High School
math directly.

## 1. Install training dependencies

In [ ]:
!pip -q install transformers datasets accelerate sentencepiece

## 2. Upload the data + training script

Run the cell, and when the file picker appears select **`train.jsonl`**,
**`val.jsonl`** (both from `ml/t5/data/`), and **`train.py`** (from `ml/t5/`).
We arrange them into the `data/` layout `train.py` expects.

In [ ]:
import os, shutil
from google.colab import files

uploaded = files.upload()  # pick train.jsonl, val.jsonl, train.py

os.makedirs('data', exist_ok=True)
for name in ('train.jsonl', 'val.jsonl'):
    if os.path.exists(name):
        shutil.move(name, os.path.join('data', name))
print('Ready:', os.listdir('.'), '| data/:', os.listdir('data'))

## 3. Confirm the GPU is attached

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'CPU only — switch runtime to T4 GPU (Runtime → Change runtime type)')

## 4. Fine-tune

Defaults: `google/flan-t5-base`, up to 20 epochs with early stopping on
validation loss (usually stops well before 20 — 140 training examples across the
four SHS subjects). The best-val checkpoint is saved to `model/`. On a T4 this
takes a few minutes and the ~1 GB base-model download won't stall like it does
on a laptop.

In [ ]:
!python train.py --data_dir data --out_dir model

## 5. Zip and download the model

Unzip the download into `A:\mathiva\ml\t5\model\` locally (replacing the
flan-t5-small smoke-test model), then evaluate with `python ml/t5/eval.py`.

In [ ]:
import shutil, os
from google.colab import files

# Bundle ONLY the files needed to run the model. The training run also leaves a
# checkpoint-*/ folder holding a duplicate model + the Adam optimizer state
# (~2-3x the model size) -- useless for inference, so we skip it. Keeps the
# download ~1 GB instead of ~4 GB.
os.makedirs('model_final', exist_ok=True)
for f in os.listdir('model'):
    p = os.path.join('model', f)
    if os.path.isfile(p):  # config, model.safetensors, tokenizer*, generation_config
        shutil.copy(p, 'model_final')

print('bundling (should be ~5 files, no checkpoint-*):', os.listdir('model_final'))
shutil.make_archive('mathiva_t5_model', 'zip', 'model_final')
files.download('mathiva_t5_model.zip')